In [1]:
from pymongo import MongoClient

client = MongoClient("127.0.0.1", 27017)
db = client["api_update"]
col = db["java_existent_api_update_instances"]
col.estimated_document_count()

57636

In [2]:
from tqdm import tqdm
import pandas as pd
from packaging.version import Version

tqdm.pandas()
commit_pairs_orig = []

for doc in tqdm(col.find({}), total=col.estimated_document_count()):
    commit = doc["commit"]
    version_before = doc["version_before"]

    version_after = doc["version_after"]
    for pair in doc["api_update_pairs"]:
        old_callee = pair["old_callee"]
        old_api_full_name = old_callee["full_name"]
        if old_callee["parameter_types"] == "":
            old_params = ""
        else:
            old_params = f"({', '.join(old_callee['parameter_types'])})"
        old_body = old_callee["body"]
        new_callee = pair["new_callee"]
        new_api_full_name = new_callee["full_name"]
        if new_callee["parameter_types"] == "":
            new_params = ""
        else:
            new_params = f"({', '.join(new_callee['parameter_types'])})"
        new_body = new_callee["body"]

        if (len(new_body) == 0) and (len(old_body) > 0):
            continue
        if (len(new_body) > 0) and (len(old_body) == 0):
            continue

        record = [
            doc["package"],
            version_before,
            version_after,
            old_api_full_name,
            old_params,
            new_api_full_name,
            new_params,
            commit,
        ]
        old_v = Version(version_before)
        new_v = Version(version_after)

        if old_v == new_v:
            continue

        # old_api_full_name, -> new_api_full_name, up
        # new_api_full_name -> old_api_full_name,, down
        elif old_v < new_v:
            if old_api_full_name < new_api_full_name:
                rule = [
                    (old_api_full_name, old_params, new_api_full_name, new_params),
                    "Up",
                ]
            else:
                rule = [
                    (new_api_full_name, new_params, old_api_full_name, old_params),
                    "Down",
                ]

        # old_api_full_name -> new_api_full_name, down
        # new_api_full_name -> old_api_full_name,, up
        else:
            if old_api_full_name < new_api_full_name:
                rule = [
                    (old_api_full_name, old_params, new_api_full_name, new_params),
                    "Down",
                ]
            else:
                rule = [
                    (new_api_full_name, new_params, old_api_full_name, old_params),
                    "Up",
                ]
        commit_pairs_orig.append(record + rule)


commit_pairs_orig = (
    pd.DataFrame(
        commit_pairs_orig,
        columns=[
            "package",
            "version_before",
            "version_after",
            "old_api_full_name",
            "old_params",
            "new_api_full_name",
            "new_params",
            "commit",
            "rule",
            "direction",
        ],
    )
    .drop_duplicates()
    .dropna()
)

print(len(commit_pairs_orig), "commit pairs before filtering")
print(
    len(commit_pairs_orig[["package", "rule", "direction"]].drop_duplicates()),
    "rules before filtering",
)
print(
    len(
        commit_pairs_orig[
            [
                "package",
                "version_before",
                "version_after",
                "old_api_full_name",
                "old_params",
                "new_api_full_name",
                "new_params",
            ]
        ].drop_duplicates()
    ),
    "pairs before filtering",
)

100%|██████████| 57636/57636 [00:13<00:00, 4141.56it/s]


50013 commit pairs before filtering
19961 rules before filtering
35179 pairs before filtering


In [3]:
def ratio_fun(row):
    if row["Down"] > row["Up"]:
        row["ratio"] = row["Down"] / row["Up"]
    else:
        row["ratio"] = row["Up"] / row["Down"]

    return row


def find_naive_error_rules():
    rule_df = (
        commit_pairs_orig.groupby(["package", "rule", "direction"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
        .rename_axis(None, axis=1)
    )
    candidate_error_rules = rule_df[(rule_df["Down"] > 0) & (rule_df["Up"] > 0)]
    print(f"{len(candidate_error_rules)} rules have both Up and Down direction")
    candidate_error_rules = candidate_error_rules.apply(ratio_fun, axis=1).sort_values(
        "ratio", ascending=False
    )
    data = []
    for row in candidate_error_rules.itertuples(index=False):
        if row.ratio < 5:
            data.append([row.package, row.rule, "Up"])
            data.append([row.package, row.rule, "Down"])
        else:
            if row.Up > row.Down:
                data.append([row.package, row.rule, "Down"])
            else:
                data.append([row.package, row.rule, "Up"])
    print(len(data), "error rules")
    error_rules = pd.DataFrame(data, columns=["package", "rule", "direction"])
    return error_rules


error_rules = find_naive_error_rules()

436 rules have both Up and Down direction
788 error rules


In [5]:
commit_pairs_filtered = (
    pd.merge(commit_pairs_orig, error_rules, indicator=True, how="left")
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
print(len(commit_pairs_filtered), "commit pairs after filtering")
print(
    len(commit_pairs_filtered[["package", "rule"]].drop_duplicates()),
    "rules after filtering",
)

47714 commit pairs after filtering
19173 rules after filtering


In [7]:
def canonical_pairs(row):
    version_before = row["version_before"]
    version_after = row["version_after"]
    package = row["package"]
    old_api_full_name = row["old_api_full_name"]
    old_params = row["old_params"]
    new_api_full_name = row["new_api_full_name"]
    new_params = row["new_params"]
    commit = row["commit"]
    if Version(version_before) > Version(version_after):
        return pd.Series(
            [
                package,
                version_after,
                version_before,
                new_api_full_name,
                new_params,
                old_api_full_name,
                old_params,
                commit,
            ],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api_full_name",
                "old_params",
                "new_api_full_name",
                "new_params",
                "commit",
            ],
        )
    else:
        return pd.Series(
            [
                package,
                version_before,
                version_after,
                old_api_full_name,
                old_params,
                new_api_full_name,
                new_params,
                commit,
            ],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api_full_name",
                "old_params",
                "new_api_full_name",
                "new_params",
                "commit",
            ],
        )


commit_pairs_full = commit_pairs_filtered.apply(canonical_pairs, axis=1)

In [8]:
commit_pairs_full.head()

,package,old_version,new_version,old_api_full_name,old_params,new_api_full_name,new_params,commit
0,junit:junit,4.8.1,4.12,junit.framework.Assert.fail,(String),org.junit.Assert.fail,(String),00001e41cb1e39747a203bd8a4454f2f1cf9005e
1,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNotNull,,org.junit.Assert.assertNotNull,,00001e41cb1e39747a203bd8a4454f2f1cf9005e
2,junit:junit,4.8.1,4.12,junit.framework.Assert.assertEquals,"(int, int)",org.junit.Assert.assertEquals,,00001e41cb1e39747a203bd8a4454f2f1cf9005e
3,junit:junit,4.8.1,4.12,junit.framework.Assert.assertNull,,org.junit.Assert.assertNull,,00001e41cb1e39747a203bd8a4454f2f1cf9005e
4,junit:junit,4.8.1,4.12,junit.framework.Assert.assertEquals,"(long, long)",org.junit.Assert.assertEquals,"(long, long)",00001e41cb1e39747a203bd8a4454f2f1cf9005e


In [9]:
rule_freq = (
    commit_pairs_full.groupby(
        [
            "package",
            "old_api_full_name",
            "old_params",
            "new_api_full_name",
            "new_params",
        ]
    )["commit"]
    .nunique()
    .reset_index()
    .sort_values("commit", ascending=False, ignore_index=True)
)
num_packages = rule_freq["package"].nunique()
num_releases = len(
    pd.concat(
        [
            commit_pairs_full[["package", "old_version"]].rename(
                columns={"old_version": "version"}
            ),
            commit_pairs_full[["package", "new_version"]].rename(
                columns={"new_version": "version"}
            ),
        ]
    ).drop_duplicates()
)
num_rules = len(rule_freq)
num_commits = commit_pairs_full["commit"].nunique()
print(f"# Packages: {num_packages}")
print(f"# Releases: {num_releases}")
print(f"# Rules: {num_rules}")
print(f"# Commits: {num_commits}")

# Packages: 2628
# Releases: 12617
# Rules: 19173
# Commits: 15949


In [10]:
pair_gte10 = rule_freq[rule_freq["commit"] >= 10]
print(
    f"{len(pair_gte10)} rules in {pair_gte10['package'].nunique()} Java packages with freq >= 10"
)
pair_1to10 = rule_freq[(rule_freq["commit"] > 1) & (rule_freq["commit"] < 10)]
print(
    f"{len(pair_1to10)} rules in {pair_1to10['package'].nunique()} Java packages with freq > 1 and < 10"
)
pair_eq1 = rule_freq[rule_freq["commit"] == 1]
print(
    f"{len(pair_eq1)} rules in {pair_eq1['package'].nunique()} Java packages with freq = 1"
)

396 rules in 96 Java packages with freq >= 10
5132 rules in 917 Java packages with freq > 1 and < 10
13645 rules in 2309 Java packages with freq = 1


In [11]:
from utils import cal_sample_size

population_size = len(pair_1to10) + len(pair_eq1)
sample_size = cal_sample_size(population_size)
sample_size_1to10 = round(sample_size * len(pair_1to10) / population_size)
sample_size_eq1 = round(sample_size * len(pair_eq1) / population_size)
print(f"Sample size for all rules with freq < 10: {sample_size}")
print(f"Sample size for rules with freq > 1 and < 10: {sample_size_1to10}")
print(f"Sample size for rules with freq = 1: {sample_size_eq1}")
pair_gte10.to_excel("../benchmark/final/java_api_update_pairs_gte10.xlsx")
pair_1to10.sample(sample_size_1to10).to_excel(
    "../benchmark/final/java_api_update_pairs_1to10.xlsx", index=False
)
pair_eq1.sample(sample_size_eq1).to_excel(
    "../benchmark/final/java_api_update_pairs_eq1.xlsx", index=False
)

Sample size for all rules with freq < 10: 376
Sample size for rules with freq > 1 and < 10: 103
Sample size for rules with freq = 1: 273


In [ ]:
pair_gte10_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_pairs_gte10-labelled.xlsx",
    keep_default_na=False,
)
correct_rules_gte10 = pair_gte10_labelled[pair_gte10_labelled["correct"] == 1]
print(
    f"{len(pair_gte10_labelled)} rules with freq >= 10, {len(correct_rules_gte10)} are correct"
)
print(f"Accuracy: {len(correct_rules_gte10) / len(pair_gte10_labelled):.3f}")

pair_1to10_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_pairs_1to10-labelled.xlsx",
    keep_default_na=False,
)
correct_rules_1to10 = pair_1to10_labelled[pair_1to10_labelled["correct"] == 1]
print(
    f"{len(pair_1to10_labelled)} rules with freq (1, 10), {len(correct_rules_1to10)} are correct"
)
print(f"Accuracy: {len(correct_rules_1to10) / len(pair_1to10_labelled):.3f}")

pair_eq1_labelled = pd.read_excel(
    "../benchmark/final/java_api_update_pairs_eq1-labelled.xlsx", keep_default_na=False
)
correct_rules_eq1 = pair_eq1_labelled[pair_eq1_labelled["correct"] == 1]
print(
    f"{len(pair_eq1_labelled)} rules with freq = 1, {len(correct_rules_eq1)} are correct"
)
print(f"Accuracy: {len(correct_rules_eq1) / len(pair_eq1_labelled):.3f}")

rules_exact = pd.concat(
    [correct_rules_gte10, correct_rules_1to10, correct_rules_eq1]
).sort_values("commit", ascending=False, ignore_index=True)
print(
    f"{len(rules_exact)} verified rule in total, {rules_exact['package'].nunique()} packages"
)
total_sampled_rules = pd.concat(
    [pair_gte10_labelled, pair_1to10_labelled, pair_eq1_labelled]
).sort_values("commit", ascending=False, ignore_index=True)
total_sampled_rules.to_csv("../benchmark/final/java_labelled_rules.csv", index=False)

396 rules with freq >= 10, 378 are correct
Accuracy: 0.955
103 rules with freq (1, 10), 90 are correct
Accuracy: 0.874
273 rules with freq = 1, 241 are correct
Accuracy: 0.883
709 verified rule in total, 261 packages


In [100]:
commit_pairs_exact = (
    rules_exact[
        [
            "package",
            "old_api_full_name",
            "old_params",
            "new_api_full_name",
            "new_params",
        ]
    ]
    .merge(commit_pairs_full)
    .drop_duplicates()
)
pairs_exact = commit_pairs_exact[
    [
        "package",
        "old_version",
        "old_api_full_name",
        "old_params",
        "new_version",
        "new_api_full_name",
        "new_params",
    ]
].drop_duplicates()
print(
    f"{len(commit_pairs_exact)} commit pairs, {len(pairs_exact)} pairs for {len(rules_exact)} correct verified rules"
)
commit_pairs_exact.to_json(
    "../benchmark/final/java_commit_pairs_exact.json", orient="records"
)
pairs_exact.to_json("../benchmark/final/java_api_pairs_exact.json", orient="records")

17051 commit pairs, 6742 pairs for 709 correct verified rules


In [103]:
new_commit_pairs_full = (
    commit_pairs_full.merge(
        total_sampled_rules[total_sampled_rules["correct"] == 0][
            [
                "package",
                "old_api_full_name",
                "old_params",
                "new_api_full_name",
                "new_params",
            ]
        ],
        indicator=True,
        how="left",
    )
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
pairs_full = new_commit_pairs_full[
    [
        "package",
        "old_version",
        "old_api_full_name",
        "old_params",
        "new_version",
        "new_api_full_name",
        "new_params",
    ]
].drop_duplicates()
rules_all = new_commit_pairs_full[
    ["package", "old_api_full_name", "old_params", "new_api_full_name", "new_params"]
].drop_duplicates()
print(
    f"{len(new_commit_pairs_full)} commit pairs, {len(pairs_full)} pairs for all {len(rules_all)} rules after removing incorrect verified rules"
)

new_commit_pairs_full.to_json(
    "../benchmark/final/java_commit_pairs_full.json", orient="records"
)
pairs_full.to_json("../benchmark/final/java_api_pairs_full.json", orient="records")

47208 commit pairs, 32518 pairs for all 19110 rules after removing incorrect verified rules


In [88]:
import pandas as pd

pairs_exact = pd.read_json("../benchmark/final/java_api_pairs_exact.json")
pairs_full = pd.read_json("../benchmark/final/java_api_pairs_full.json")
len(pairs_exact), len(pairs_full)

(3769, 32632)

In [89]:
def get_update_type(row):
    result = ["Major", "Minor", "Patch"]
    old_version = row["old_version"].split(".")
    new_version = row["new_version"].split(".")
    min_len = min(len(old_version), len(new_version))
    row["major"] = (int(old_version[0]), int(new_version[0]))
    for i in range(min_len):
        if old_version[i] != new_version[i]:
            row["update_type"] = result[i]
            return row
    row["update_type"] = result[min_len]
    return row

In [104]:
pairs_exact_sample = (
    pairs_exact.apply(get_update_type, axis=1)
    .groupby(
        [
            "package",
            "old_api_full_name",
            "old_params",
            "new_api_full_name",
            "new_params",
            "major",
            "update_type",
        ]
    )
    .sample(1)
)[
    [
        "package",
        "old_version",
        "old_api_full_name",
        "old_params",
        "new_version",
        "new_api_full_name",
        "new_params",
    ]
]
print(f"{len(pairs_exact_sample)} sampled pairs in the exact group")
pairs_exact_sample.to_json(
    "../benchmark/final/java_sampled_api_pairs_exact.json", orient="records"
)

pairs_full_sample = (
    pairs_full.apply(get_update_type, axis=1)
    .groupby(
        [
            "package",
            "old_api_full_name",
            "old_params",
            "new_api_full_name",
            "new_params",
            "major",
            "update_type",
        ]
    )
    .sample(1)
)[
    [
        "package",
        "old_version",
        "old_api_full_name",
        "old_params",
        "new_version",
        "new_api_full_name",
        "new_params",
    ]
]
print(f"{len(pairs_full_sample)} sampled pairs in the full group")
pairs_full_sample.to_json(
    "../benchmark/final/java_sampled_api_pairs_full.json", orient="records"
)

1385 sampled pairs in the exact group
21589 sampled pairs in the full group
